# Drive → Kaggle: gemma4:e4b model dosyalarını aktar

Amaç: Google Drive'daki `ollama_models` klasörünü (gemma4:e4b'nin model dosyaları)
senin bilgisayarına hiç indirmeden, doğrudan Colab üzerinden Kaggle'a bir Dataset
olarak yükler. İki bulut servisi arasındaki bağlantı, senin ev internetinden çok
daha hızlı/güvenilir olduğu için büyük klasör indirme sorununu bypass eder.

**Bu, çalışan hakem batch notebook'undan (`colab_run_gemma_judge_reversed.ipynb`)
tamamen ayrı, bağımsız bir notebook** — ona dokunmuyoruz, yeni bir Colab çalışma
zamanında (runtime) çalıştırılmalı.

**Güvenlik notu:** Kaggle API token'ı bu dosyaya hiç yazılmıyor, KAGGLE_USERNAME
ise gizli değil (herkese açık profil adı). Token'ı 3. hücre çalışınca gizli
(görünmez) bir giriş kutusunda gireceksin - dosyada/repo'da hiçbir zaman durmaz.

Bir kerelik bir işlem — çalıştırdıktan sonra bu notebook'u kapatabilirsin.

## 1) Google Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OLLAMA = "/content/drive/MyDrive/institution_resolver_v3/ollama_models"

import os
assert os.path.isdir(DRIVE_OLLAMA), f"Bulunamadı: {DRIVE_OLLAMA} - Drive'daki klasör yolunu kontrol edin."
print("Bulundu:", DRIVE_OLLAMA)
!du -sh "{DRIVE_OLLAMA}"

## 2) Kaggle bilgilerinizi girin

In [ ]:
import getpass

KAGGLE_USERNAME = input("Kaggle kullanıcı adınız (kaggle.com/<bu-kısım>): ").strip()
KAGGLE_API_TOKEN = getpass.getpass("Kaggle API token'ınız (KGAT_... ile başlayan, görünmez girilir): ").strip()

assert KAGGLE_USERNAME, "Kullanıcı adı boş olamaz"
assert KAGGLE_API_TOKEN.startswith("KGAT_"), "Token 'KGAT_' ile başlamalı - doğru kopyaladığınızdan emin olun"
print("Bilgiler alındı (token gösterilmiyor).")

## 3) Kaggle CLI kurulumu + kimlik doğrulama

In [ ]:
!pip install -q kaggle

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(KAGGLE_API_TOKEN)
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)

!kaggle datasets list --user "{KAGGLE_USERNAME}" 2>&1 | head -5
print("\n(Yukarıda bir hata yerine liste/boş sonuç görüyorsanız kimlik doğrulama çalışıyor demektir.)")

## 4) Dataset oluştur ve Drive'dan doğrudan Kaggle'a push et

Bu hücre, `ollama_models` klasörünü Drive'da bulunduğu yerden okuyup zip'leyerek
Kaggle'a yükler - hiçbir aşamada sizin bilgisayarınızın diskine/internetine uğramaz.

In [ ]:
import json, shutil, tempfile, os

DATASET_SLUG = "gemma4-e4b-ollama"

# Drive klasörü salt-okunur/paylaşımlı olabileceği için metadata dosyasını
# ayrı bir geçici klasöre kopyalayıp oradan yüklüyoruz (Drive'a yazmıyoruz).
staging = "/content/kaggle_model_staging"
if os.path.isdir(staging):
    shutil.rmtree(staging)
shutil.copytree(DRIVE_OLLAMA, staging)

meta = {
    "title": DATASET_SLUG,
    "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
}
with open(os.path.join(staging, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f)

print(f"'{DATASET_SLUG}' Kaggle Dataset'i olarak yükleniyor (birkaç dakika sürebilir)...")
!kaggle datasets create -p "{staging}" -r zip

print(f"\nBitince kontrol edin: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")

## 5) needs_review_subset.csv'yi (TAM dosya, A→Z) de aynı şekilde yükle

2026-08-13: plan güncellendi - 3'e bölünmüş dilim yerine artık Kaggle TAM dosyayı
A'dan başlayarak işliyor (Colab zaten aynı tam dosyayı Z'den işliyor, dokunulmuyor).
Bu dosya `data/jobs/needs_review_subset.csv`'nin Drive'a yüklenmiş hali olmalı.

In [ ]:
import json, shutil, os

CSV_DATASET_SLUG = "needs-review-full"
DRIVE_CSV = "/content/drive/MyDrive/institution_resolver_v3/jobs/needs_review_subset.csv"

assert os.path.isfile(DRIVE_CSV), f"Bulunamadı: {DRIVE_CSV} - önce bu dosyayı Drive'a yüklediniz mi?"

staging_csv = "/content/kaggle_csv_staging"
os.makedirs(staging_csv, exist_ok=True)
shutil.copy(DRIVE_CSV, staging_csv)

meta = {
    "title": CSV_DATASET_SLUG,
    "id": f"{KAGGLE_USERNAME}/{CSV_DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
}
with open(os.path.join(staging_csv, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f)

!kaggle datasets create -p "{staging_csv}"
print(f"\nBitince kontrol edin: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{CSV_DATASET_SLUG}")

## 6) (Opsiyonel) Boş çıktı Dataset'i oluştur

`kaggle_run_gemma_judge.ipynb`'daki ara-yedekleme fonksiyonunun push edeceği
hedef - içine anlamsız bir placeholder dosya koyup boş bir Dataset yaratıyoruz,
asıl hakem koşusu sırasında üzerine gerçek sonuçlar yazılacak.

In [ ]:
import json, os

OUTPUT_DATASET_SLUG = "kaggle-judge-output"
staging_out = "/content/kaggle_output_staging"
os.makedirs(staging_out, exist_ok=True)

with open(os.path.join(staging_out, "placeholder.txt"), "w") as f:
    f.write("ilk yedek burada oluşacak")

meta = {
    "title": OUTPUT_DATASET_SLUG,
    "id": f"{KAGGLE_USERNAME}/{OUTPUT_DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
}
with open(os.path.join(staging_out, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f)

!kaggle datasets create -p "{staging_out}"
print(f"\nBitince kontrol edin: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{OUTPUT_DATASET_SLUG}")

---
## Bitti — sırada ne var

Üç Dataset de oluştuğuna göre, `kaggle_run_gemma_judge.ipynb` notebook'unu Kaggle'a
yükleyip 2. hücresindeki `KAGGLE_USERNAME` ve dataset yollarını kendi bilgilerinizle
doldurun, sağ panelden üç dataset'i de "Add Input" ile ekleyin, Secrets'a
`KAGGLE_API_TOKEN`'ı girin, ve hücreleri sırayla çalıştırın.